# 01 - Data Collection (FastF1)

Replaces the manually-collected `data/race_data.csv` with data pulled directly from the F1 timing API via [FastF1](https://docs.fastf1.dev/).

**Output schema matches the existing manual dataset:**
`Driver, GP, StartType, StartPos, FinishPos, Delta, Stops, FirstPitLap, Status`

One change: penalty/annotation notes that used to live inside `FirstPitLap` (e.g. `"10 (5s penalty served)"`) are now split into a separate `Notes` column, so `FirstPitLap` stays numeric and machine-readable. See the note at the bottom.

> Run `pip install fastf1` first if you haven't already.

In [1]:
import fastf1
import pandas as pd
import numpy as np
import os

# FastF1 caches downloaded sessions locally so re-runs are fast and don't
# re-hit the API. Point this at a folder in your repo (add it to .gitignore).
CACHE_DIR = "../fastf1_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
fastf1.Cache.enable_cache(CACHE_DIR)

## Config

Add/remove Grand Prix here to expand your circuit set later without touching the fetch logic.

In [2]:
YEAR = 2025
GRANDS_PRIX = ["Bahrain", "Monza", "Silverstone", "Monaco"]

## Status mapping

FastF1 reports status in English (`Finished`, `Retired`, `Disqualified`, `+1 Lap`, etc). Your existing CSV uses Italian labels for the finish-position field on non-finishers (`Ritirato`, `Squalificato`) and `DNF`/`DSQ` in the `Status` column. This keeps that convention so old and new rows are consistent — change `STATUS_LABELS` if you'd rather standardize to English going forward (probably a good idea for the final thesis version).

In [4]:
STATUS_LABELS = {
    "Finished": ("Finished", None),        # (Status value, FinishPos override)
    "Retired": ("DNF", "Ritirato"),
    "Disqualified": ("DSQ", "Squalificato"),
    "Did not start": ("DNS", "DNS"),
}

def map_status(raw_status: str):
    """Return (Status, FinishPos override or None) for a FastF1 status string.
    Anything not explicitly listed (e.g. '+1 Lap') is treated as a normal finish."""
    if raw_status in STATUS_LABELS:
        return STATUS_LABELS[raw_status]
    return ("Finished", None)

## Fetch one race

In [5]:
def format_delta(grid_pos: float, finish_pos: float) -> str:
    """Match the existing '+N' / '-N' / '0' string format for Delta."""
    delta = int(grid_pos - finish_pos)
    if delta > 0:
        return f"+{delta}"
    return str(delta)


def get_race_data(year: int, gp_name: str) -> pd.DataFrame:
    session = fastf1.get_session(year, gp_name, "R")
    session.load()

    results = session.results[
        ["Abbreviation", "GridPosition", "Position", "Status"]
    ].copy()

    laps = session.laps
    pit_laps = laps[laps["PitInTime"].notna()]
    pit_summary = (
        pit_laps.groupby("Driver")
        .agg(Stops=("LapNumber", "count"), FirstPitLap=("LapNumber", "min"))
        .reset_index()
        .rename(columns={"Driver": "Abbreviation"})
    )

    df = results.merge(pit_summary, on="Abbreviation", how="left")

    rows = []
    for _, r in df.iterrows():
        status_val, finish_override = map_status(r["Status"])
        grid_pos = r["GridPosition"]
        finish_pos_numeric = r["Position"]

        # Pit lane starts show up as GridPosition == 0 in FastF1
        start_type = "Pit Lane" if grid_pos == 0 else "Grid"

        if finish_override is not None:
            finish_pos = finish_override
            delta = "NA"
        else:
            finish_pos = int(finish_pos_numeric)
            delta = format_delta(grid_pos, finish_pos_numeric)

        rows.append({
            "Driver": r["Abbreviation"],
            "GP": gp_name,
            "StartType": start_type,
            "StartPos": int(grid_pos),
            "FinishPos": finish_pos,
            "Delta": delta,
            "Stops": int(r["Stops"]) if pd.notna(r["Stops"]) else 0,
            "FirstPitLap": int(r["FirstPitLap"]) if pd.notna(r["FirstPitLap"]) else np.nan,
            "Status": status_val,
            "Notes": "",  # fill in manually for penalties served, investigations, etc.
        })

    return pd.DataFrame(rows)

## Fetch all races and save

In [6]:
all_data = pd.concat(
    [get_race_data(YEAR, gp) for gp in GRANDS_PRIX],
    ignore_index=True,
)

all_data.head(20)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca

,Driver,GP,StartType,StartPos,FinishPos,Delta,Stops,FirstPitLap,Status,Notes
0,PIA,Bahrain,Grid,1,1,0,2,14.0,Finished,
1,RUS,Bahrain,Grid,3,2,+1,2,13.0,Finished,
2,NOR,Bahrain,Grid,6,3,+3,2,10.0,Finished,
3,LEC,Bahrain,Grid,2,4,-2,2,17.0,Finished,
4,HAM,Bahrain,Grid,9,5,+4,2,17.0,Finished,
5,VER,Bahrain,Grid,7,6,+1,2,10.0,Finished,
6,GAS,Bahrain,Grid,4,7,-3,2,10.0,Finished,
7,OCO,Bahrain,Grid,14,8,+6,2,8.0,Finished,
8,TSU,Bahrain,Grid,10,9,+1,2,11.0,Finished,
9,BEA,Bahrain,Grid,20,10,+10,2,14.0,Finished,


In [7]:
OUTPUT_PATH = "../data/race_data_api.csv"
all_data.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(all_data)} rows to {OUTPUT_PATH}")

Saved 80 rows to ../data/race_data_api.csv


## Notes / things to check before you fully switch over

- **`Notes` column is new** — it's empty by default. Penalty details (e.g. "5s penalty served") aren't part of FastF1's `Status`/lap data in a clean structured way; you'll still need to add those manually, but now they live in their own column instead of being crammed into `FirstPitLap`.
- **`FirstPitLap` is now purely numeric.** Your old `"14 (44 to check)"` style entries won't reproduce — if that was flagging a data-quality doubt, the API removes that doubt (FastF1 pit lap numbers come straight from official timing).
- **Compare, don't blindly overwrite.** I saved this to `race_data_api.csv` (not overwriting `race_data.csv`) so you can diff the two and confirm the API version agrees with your manually-collected races before making it the source of truth.
- **Pit lane starts**: FastF1 reports these as `GridPosition == 0`. Worth spot-checking against results pages for Bahrain/Monza/Silverstone/Monaco since it affects your `Delta` calculation.
- **Add `fastf1_cache/` to `.gitignore`** — it can get large and doesn't belong in version control.